# Top-K activating images for SAE latents

For every SAE latent kept after pruning, we show the K training images that activate it most strongly.

- The **heatmap** projects the backbone's last feature map (`layer4`, 7×7) onto the latent's decoder direction. Bright regions are where the latent's direction is present; everything else is dimmed.
- Under each image: its class and the latent's activation value.
- The bar chart shows which classes dominate the top activations. A latent spread over many classes is more likely a shared visual concept; one dominated by a single class may just be a class detector.

Run the pipeline first (see `README.md`). By default this notebook uses the most recent SAE run in `outputs/sae/`.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import Image, display

from concept_learning.topk import LatentExplorer, latest_run

SAE_DIR = latest_run("sae")
explorer = LatentExplorer(SAE_DIR)
print(f"SAE run: {SAE_DIR.name}")
print(f"Backbone: {explorer.activation_info['backbone']}")

## Run summary

In [ ]:
m = explorer.metrics
p = m["pruning"]
summary = {
    "classifier accuracy (train / val / test)": " / ".join(f"{explorer.activation_info['classifier_accuracy'][s]:.2%}" for s in ["train", "val", "test"]),
    "SAE latents": m["latent_dim"],
    "kept after pruning": p["kept_units"],
    "val reconstruction MSE": f"{m['val_reconstruction_loss']:.5f}",
    "val avg L0 (active latents per image)": f"{m['val_avg_l0']:.2f}",
    "recovered accuracy, all latents": f"{p['baseline_recovered_accuracy']:.2%}",
    "recovered accuracy, kept latents": f"{p['selected_recovered_accuracy']:.2%}",
}
for key, value in summary.items():
    print(f"{key:>42}: {value}")

In [ ]:
for plot in ["training_curves", "feature_density", "pruning_curve"]:
    display(Image(filename=SAE_DIR / "plots" / f"{plot}.png", width=900))

## Browse latents

Interactive picker. It only works in a live kernel; GitHub doesn't render widgets, so the static gallery below is what shows up in the committed notebook.

In [ ]:
latent_picker = widgets.Dropdown(
    options=[(f"latent {u} (fires on {int(explorer.unit_counts[u])} images)", u) for u in explorer.latents_by_frequency()],
    description="Latent",
    layout=widgets.Layout(width="420px"),
)
k_slider = widgets.IntSlider(value=10, min=5, max=30, step=5, description="K")
heatmap_toggle = widgets.Checkbox(value=True, description="Heatmap")


def show(latent, k, heatmap):
    explorer.plot_latent(latent, k=k, heatmap=heatmap)
    plt.show()


output = widgets.interactive_output(show, {"latent": latent_picker, "k": k_slider, "heatmap": heatmap_toggle})
display(widgets.HBox([latent_picker, k_slider, heatmap_toggle]), output)

## Static gallery

The `N_LATENTS` most frequently firing kept latents, plus a random sample of the rest, so rarer (often more specific) latents get shown too.

In [ ]:
import random

N_LATENTS = 10
N_RANDOM = 10

by_frequency = explorer.latents_by_frequency()
gallery = by_frequency[:N_LATENTS] + sorted(random.Random(0).sample(by_frequency[N_LATENTS:], min(N_RANDOM, len(by_frequency) - N_LATENTS)))
for latent in gallery:
    explorer.plot_latent(latent, k=10)
    plt.show()